In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from huggingface_hub import InferenceClient

from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

api_key = os.getenv('HUGGINGFACE_API_KEY')
if not api_key:
    raise ValueError("HUGGINGFACE_API_KEY not found in .env file. Please add your API key.")
print(f"API key loaded: {api_key[:10]}...")


API key loaded: hf_OnJvVQX...


In [3]:
llm = HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-7B-Instruct',
    huggingfacehub_api_token=api_key
) 

In [4]:
model =ChatHuggingFace(llm=llm)

In [5]:
# define state
class LLMstate(TypedDict):
    question : str
    answer : str

In [6]:
# define graph
graph  = StateGraph(LLMstate)


In [7]:
def Llm_call_qa(state :LLMstate )-> LLMstate:
    # extract question from state 
     quest = state['question']
    # preapred prompt and send to model
     prompt = f'provide the appropriate answer for this {quest}'
    # extarct ans 
     answer = model.invoke(prompt).content
# set answer to state
     state['answer'] = answer
     return state

In [8]:
# define nodes 
if 'Llm_call_qa' not in graph.nodes:
    graph.add_node('Llm_call_qa', Llm_call_qa)

# define edge
graph.add_edge(START,'Llm_call_qa')
graph.add_edge('Llm_call_qa',END)

In [9]:
# compile graph
workflow= graph.compile()
initial_state = {'question':'what is kidlins law or problem solving'}
final_state = workflow.invoke(initial_state)
print(final_state)


{'question': 'what is kidlins law or problem solving', 'answer': "It appears there might be a typo in your query. I suspect you're referring to **Kipling's Law** or **Kiedrowski's Law**, but neither term is commonly used in the context of problem-solving. However, if you're referring to a concept related to problem-solving, there are a few well-known principles that might be what you're looking for. Could you please clarify if you're referring to something specific? \n\nIf you meant:\n\n1. **Kahneman's Heuristics and Biases** - This is a principle in psychology that discusses how people make decisions and solve problems using heuristics (rules of thumb), which can sometimes lead to biases.\n2. **Kepner-Tregoe Problem-Solving Method** - This is a systematic approach to problem solving that involves a decision-making process.\n3. **Klutz's Law** - Often humorously attributed to Tom Klutz, it states that when you're in a hurry, things that can go wrong, will go wrong.\n4. **Kolb's Learnin